In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

# Moderation Effect Analysis: Sex

In [ ]:
# Define confounders as a string
confounders <- c("age + SEX + bmi + f_wnowt")

# isolate metabolite names
met_names <- rownames(D)

First, we'll subset our maplet object to isolate RV-specific metabolite profiles that are non-missing. 

In [ ]:
D1_GLS <- D %>% mt_modify_filter_samples(filter = !is.na(RVGLOB6n)) # exclude samples with missing RVGLOB6n parameters

In [ ]:
D1_FAC <- D %>% mt_modify_filter_samples(filter = !is.na(RVFACn)) # exclude samples with missing RVFACn parameters

In [ ]:
D1_RVEF <- D %>% mt_modify_filter_samples(filter = !is.na(mri_RVEF)) # exclude samples with missing mri_RVEF parameters

## TODO: Store these interaction df as helpers 

In [ ]:
# function to prepare a df of mets, and clin_var
interaction_df_prep <- function(D, clin_var) { 
    clin_df <- D %>% colData() %>% as.data.frame() %>% 
        # select the clinical variable
        select(!!sym(clin_var), age, SEX, bmi, f_wnowt) %>% tibble::rownames_to_column('StudyID') %>% na.omit()

    assay_df <- D %>% assay() %>% t() %>% as.data.frame() %>% tibble::rownames_to_column('StudyID')

    # join these by StudyID
    joined_df <- clin_df %>% inner_join(assay_df, by = 'StudyID') %>% 
        tibble::column_to_rownames('StudyID')
}

In [ ]:
library(broom)
library(dplyr)
library(stringr)
library(purrr)

run_moderation_models_sex <- function(data, clin_var, met_names, covars = NULL) {

  # Ensure SEX is present, is numeric, and binary [0,1]; stop otherwise
  if (!("SEX" %in% names(data))) stop("SEX column not found in data.")
  if (!is.numeric(data$SEX) && !is.integer(data$SEX)) stop("SEX must be numeric/integer.")
  unique_sex <- sort(unique(na.omit(data$SEX)))
  if (!all(unique_sex %in% c(0,1)) || length(unique_sex) > 2) stop("SEX must take only two values, 0 (Female) and 1 (Male).")
  
  results_list <- list()
  
  for (met in met_names) {
    
    # Construct formula with interaction: clin_var*SEX
    interaction_term <- paste0(clin_var, "*SEX")
    covar_str <- if (!is.null(covars) && length(covars) > 0) paste(covars, collapse = " + ") else NULL
    rhs <- paste(c(interaction_term, covar_str), collapse = " + ")
    formula_str <- paste0("`", met, "` ~ ", rhs)
    fmla <- as.formula(formula_str)
    
    # Fit the model
    model <- lm(fmla, data = data)
    
    # Tidy the results
    tidy_results <- broom::tidy(model)
    
    # Add metadata
    tidy_results <- tidy_results %>%
      mutate(
        met_name = met,
        # Coarse label
        coefficient_type = case_when(
          term == "(Intercept)" ~ "Intercept",
          term == clin_var ~ "ClinicalVar",
          term == "SEX" ~ "Sex_Male_Baseline",  # main effect of SEX=1
          str_detect(term, ":") ~ "Interaction",
          TRUE ~ "Other"
        ),
        # Fine-grained (SEX=1 = Male, 0 = Female; interactions)
        coefficient_type_2 = case_when(
          term == "(Intercept)" ~ "Intercept",
          term == clin_var ~ "ClinicalVar",
          term == "SEX" ~ "Male_Baseline", # effect of being Male when clin_var==0
          term == paste0(clin_var, ":SEX") | term == paste0("SEX:", clin_var) ~ "Male_Interaction",
          term == "age" ~ "Other_age",
          term == "bmi" ~ "Other_bmi",
          TRUE ~ "Other"
        )
      )
    
    # Store in list
    results_list[[met]] <- tidy_results
  }
  
  # Combine all results
  results_df <- bind_rows(results_list)
  return(results_df)
}

## GLS 6`

In [ ]:
var_of_interest <- "RVGLOB6n" # define rv parameter

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_GLS, clin_var = var_of_interest)

results_df <- run_moderation_models_sex(
  data = interaction_df, #  df containing sex, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_interaction.xlsx"))))

## FAC

In [ ]:
var_of_interest <- "RVFACn" # define rv parameter

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_FAC, clin_var = var_of_interest)

results_df <- run_moderation_models_sex(
  data = interaction_df, #  df containing sex, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_interaction.xlsx"))))

## RVEF

In [ ]:
var_of_interest <- "mri_RVEF" # define rv parameter

# prepare df to interaction term anaylsis
interaction_df <- interaction_df_prep(D1_RVEF, clin_var = var_of_interest)

results_df <- run_moderation_models_sex(
  data = interaction_df, #  df containing sex, metabolite profiles, and clinical confounders
  clin_var = var_of_interest, # rv parameter
  met_names = met_names, # metabolite names
  covars = confounders # confounders
)

In [ ]:
# prepare table for export (main effect and interaction term)
stats_main <- results_df %>% filter(coefficient_type == 'ClinicalVar') %>% mutate(FDR = p.adjust(p.value, method = 'fdr')) 
stats_interaction <- results_df %>% filter(coefficient_type == 'Interaction') %>% mutate(FDR = p.adjust(p.value, method = 'fdr'))

# export
write.csv(stats_main, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_main.xlsx"))))
write.csv(stats_interaction, file.path(here('outputs', paste0(var_of_interest, "_sex_moderation_interaction.xlsx"))))

# TODO: Confirm results align with that reported 